# ToolNode 与 ToolRuntime：工具调用执行和状态更新

> 适用版本：本项目锁定的 **LangGraph 1.1.2**、**langgraph-prebuilt 1.0.8**、**LangChain 1.2.12**、**langchain-core 1.2.18**、**langchain-deepseek 1.0.1**。

本教程的前两节会读取 `.env` 并真实调用 **DeepSeek Flash**（`deepseek:deepseek-v4-flash`），由模型生成带 `tool_calls` 的 `AIMessage`，再分别演示手动执行与 `ToolNode` 执行。后续 `ToolRuntime`、`Command` 和 reducer 边界验证仍使用本地确定性工具，不依赖模型输出。真实模型的调用 ID、消息 ID 和自然语言措辞每次可能不同，因此断言只验证稳定的结构与计算结果。

学习目标：

1. 手动完成“识别调用 → 分发工具 → 执行 → 封装 `ToolMessage` → 写回消息 → 继续模型步骤”；
2. 理解 `ToolNode` 替代了哪些样板代码，以及它**不负责调用模型**；
3. 使用 `ToolRuntime` 读取 state/context/store/config/tool_call_id，并用 `Command` 更新图状态；
4. 用实际运行和源码断言澄清：`Command(update=...)` 何时会、何时不会产生 `ToolMessage`；
5. 理解 `add_messages` reducer 与 `tool_call_id` 的配对约束。

## 总体流程

```mermaid
flowchart LR
    U[HumanMessage] --> M1[模型步骤]
    M1 -->|AIMessage.tool_calls| D{是否有工具调用}
    D -->|否| E[结束]
    D -->|是| T[手动分发或 ToolNode]
    T --> X[执行工具]
    X --> TM[生成 ToolMessage
匹配 tool_call_id]
    TM --> R[add_messages 写回 state.messages]
    R --> M2[下一次模型步骤]
```

`ToolNode` 位于虚线概念中的“工具执行侧”：它读取最后一条 `AIMessage` 的 `tool_calls` 并执行工具。调用模型、决定是否调用工具、以及收到工具结果后继续生成回答，仍由模型节点或应用代码负责。

In [21]:
import importlib.metadata as metadata
import inspect
import json
import operator
import os
from dataclasses import dataclass
from pprint import pprint
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import ToolRuntime
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph, add_messages
from langgraph.prebuilt import ToolNode
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command


versions = {
    package: metadata.version(package)
    for package in ("langgraph", "langgraph-prebuilt", "langchain", "langchain-core", "langchain-deepseek")
}
print("当前实际运行版本（原始字典）：")
pprint(versions)

assert versions == {
    "langgraph": "1.1.2",
    "langgraph-prebuilt": "1.0.8",
    "langchain": "1.2.12",
    "langchain-core": "1.2.18",
    "langchain-deepseek": "1.0.1",
}

load_dotenv(override=True)
deepseek_key_configured = bool(os.getenv("DEEPSEEK_API_KEY"))
print("DEEPSEEK_API_KEY:", "SET（值已隐藏）" if deepseek_key_configured else "BLOCKED（未配置）")
if not deepseek_key_configured:
    raise RuntimeError("未配置 DEEPSEEK_API_KEY，无法运行真实模型示例。")

DEEPSEEK_MODEL = "deepseek:deepseek-v4-flash"
deepseek_model = init_chat_model(
    model=DEEPSEEK_MODEL,
    temperature=0,
    extra_body={"thinking": {"type": "disabled"}},
)
print("真实模型客户端：", DEEPSEEK_MODEL)

当前实际运行版本（原始字典）：
{'langchain': '1.2.12',
 'langchain-core': '1.2.18',
 'langchain-deepseek': '1.0.1',
 'langgraph': '1.1.2',
 'langgraph-prebuilt': '1.0.8'}
DEEPSEEK_API_KEY: SET（值已隐藏）
真实模型客户端： deepseek:deepseek-v4-flash


## 1. 手动执行工具调用

模型通常只负责在 `AIMessage.tool_calls` 中提出结构化调用请求；真正执行 Python 工具的是应用。下面把 `add` 和 `multiply` 绑定到真实 DeepSeek Flash：

- 第一次请求明确要求模型调用两个工具，得到真实的 `AIMessage.tool_calls`；
- 应用手动执行工具并写回两个 `ToolMessage`；
- 第二次请求把完整消息历史交回模型，由模型根据工具结果生成最终回答。

手动实现时，应用必须自己维护工具注册表，并为每个调用创建 `ToolMessage`。其中 `ToolMessage.tool_call_id` 必须等于对应调用的 `id`。

In [22]:
class ManualState(TypedDict):
    messages: Annotated[list, add_messages]


@tool
def add(a: int, b: int) -> int:
    """返回两个整数之和。"""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """返回两个整数之积。"""
    return a * b


manual_tools = {item.name: item for item in (add, multiply)}
tool_calling_model = deepseek_model.bind_tools([add, multiply])
print("已绑定给真实模型的工具：", list(manual_tools))

已绑定给真实模型的工具： ['add', 'multiply']


In [23]:
manual_state: ManualState = {
    "messages": [
        HumanMessage(
            id="human-001",
            content=(
                "请务必调用 add 工具计算 2 + 3，并调用 multiply 工具计算 4 × 5。"
                "两个工具都必须调用，不要自己心算；获得工具结果后再用中文汇总。"
            ),
        )
    ]
}

print("步骤 0：初始 state（完整原始对象）")
pprint(manual_state)

first_ai_message = tool_calling_model.invoke(manual_state["messages"])
manual_state["messages"] = add_messages(manual_state["messages"], [first_ai_message])

print("\n步骤 1：DeepSeek Flash 返回的完整 AIMessage")
pprint(first_ai_message)
print("\n步骤 1：写回后的完整 messages")
pprint(manual_state["messages"])
print("\n结构化 tool_calls 原始字段")
pprint(first_ai_message.tool_calls)

步骤 0：初始 state（完整原始对象）
{'messages': [HumanMessage(content='请务必调用 add 工具计算 2 + 3，并调用 multiply 工具计算 4 × 5。两个工具都必须调用，不要自己心算；获得工具结果后再用中文汇总。', additional_kwargs={}, response_metadata={}, id='human-001')]}

步骤 1：DeepSeek Flash 返回的完整 AIMessage
AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 385, 'total_tokens': 489, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 1}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'cd11239c-7e68-4b68-844c-cb2a8ac48988', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0343b-6f1f-76c2-a569-51b6267d276e-0', tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'call_00_ZRrAvbO8krCFgYKZUsUJ7560', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 4, 'b'

In [24]:
# 应用代码读取最后一条 AIMessage，并逐个分发工具。
last_ai_message = manual_state["messages"][-1]
created_tool_messages: list[ToolMessage] = []

for call in last_ai_message.tool_calls:
    selected_tool = manual_tools[call["name"]]
    raw_result = selected_tool.invoke(call["args"])

    # 手动执行时，调用方必须负责创建匹配调用 ID 的 ToolMessage。
    tool_message = ToolMessage(
        content=json.dumps(raw_result, ensure_ascii=False),
        name=call["name"],
        tool_call_id=call["id"],
    )
    created_tool_messages.append(tool_message)

    print(f"\n执行调用 {call['id']}：完整调用、原始返回值、ToolMessage")
    pprint(call)
    pprint(raw_result)
    pprint(tool_message)

    manual_state["messages"] = add_messages(manual_state["messages"], [tool_message])
    print("当前完整 messages：")
    pprint(manual_state["messages"])

print("\n步骤 2：两个工具都执行后的完整 state")
pprint(manual_state)


执行调用 call_00_ZRrAvbO8krCFgYKZUsUJ7560：完整调用、原始返回值、ToolMessage
{'args': {'a': 2, 'b': 3},
 'id': 'call_00_ZRrAvbO8krCFgYKZUsUJ7560',
 'name': 'add',
 'type': 'tool_call'}
5
ToolMessage(content='5', name='add', tool_call_id='call_00_ZRrAvbO8krCFgYKZUsUJ7560')
当前完整 messages：
[HumanMessage(content='请务必调用 add 工具计算 2 + 3，并调用 multiply 工具计算 4 × 5。两个工具都必须调用，不要自己心算；获得工具结果后再用中文汇总。', additional_kwargs={}, response_metadata={}, id='human-001'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 385, 'total_tokens': 489, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 1}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'cd11239c-7e68-4b68-844c-cb2a8ac48988', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run-

In [25]:
# 工具结果已进入消息历史，现在才继续“下一次模型步骤”。
final_ai_message = tool_calling_model.invoke(manual_state["messages"])
manual_state["messages"] = add_messages(manual_state["messages"], [final_ai_message])

print("步骤 3：下一模型步骤的完整 AIMessage")
pprint(final_ai_message)
print("\n最终完整 state")
pprint(manual_state)
print("\n可选友好摘要：", final_ai_message.content)

步骤 3：下一模型步骤的完整 AIMessage
AIMessage(content='根据工具计算结果：\n\n1. **add 工具**计算 2 + 3，结果为 **5**。\n2. **multiply 工具**计算 4 × 5，结果为 **20**。\n\n汇总：2 加 3 等于 5，4 乘以 5 等于 20。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 511, 'total_tokens': 576, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 127}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'c5708730-8a4e-4656-be22-bc38e4dfc915', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0343b-72cd-7990-b8dd-3d223bdf2246-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 511, 'output_tokens': 65, 'total_tokens': 576, 'input_token_details': {'cache_read': 384}, 'output_token_details': {}})

最终完整 state
{'messages': [HumanMessage(content='请务必调用 add 工具计算 2 + 3，

In [26]:
# 机制断言与主示例分开：逐一验证工具调用和 ToolMessage 的配对关系。
requested_ids = [call["id"] for call in first_ai_message.tool_calls]
returned_ids = [message.tool_call_id for message in created_tool_messages]
requested_by_name = {call["name"]: call for call in first_ai_message.tool_calls}
returned_values = {message.name: json.loads(message.content) for message in created_tool_messages}

print("请求 ID（原始列表）：")
pprint(requested_ids)
print("返回 ID（原始列表）：")
pprint(returned_ids)

assert set(requested_by_name) == {"add", "multiply"}
assert requested_by_name["add"]["args"] == {"a": 2, "b": 3}
assert requested_by_name["multiply"]["args"] == {"a": 4, "b": 5}
assert returned_ids == requested_ids
assert returned_values == {"add": 5, "multiply": 20}
assert first_ai_message.response_metadata.get("model_name") == "deepseek-v4-flash"
assert not final_ai_message.tool_calls
assert "5" in final_ai_message.content and "20" in final_ai_message.content
print("全部手动执行断言通过。")

请求 ID（原始列表）：
['call_00_ZRrAvbO8krCFgYKZUsUJ7560', 'call_01_6s6eEf0hYnPiuDuRXb5k9770']
返回 ID（原始列表）：
['call_00_ZRrAvbO8krCFgYKZUsUJ7560', 'call_01_6s6eEf0hYnPiuDuRXb5k9770']
全部手动执行断言通过。


## 2. 用 ToolNode 替代手动分发、调用、封装和写回

`ToolNode([add, multiply])` 会读取 state 中最后一条 `AIMessage`，并对其中每个 `tool_call`：

1. 按名称查找工具；
2. 将 `args` 交给工具；
3. 对普通返回值创建 `ToolMessage`；
4. 将结果作为 `{"messages": [...]}` 节点更新返回，随后由 `add_messages` 合并进状态。

这里构建的图**只有 ToolNode，没有模型节点**。因此运行结果以 `ToolMessage` 结束，能直接证明 ToolNode 不会继续调用模型。

In [27]:
class ToolNodeState(TypedDict):
    messages: Annotated[list, add_messages]


tool_builder = StateGraph(ToolNodeState)
tool_builder.add_node("tools", ToolNode([add, multiply]))
tool_builder.add_edge(START, "tools")
tool_builder.add_edge("tools", END)
tool_graph = tool_builder.compile()

toolnode_input: ToolNodeState = {
    "messages": [
        HumanMessage(id="human-toolnode", content=manual_state["messages"][0].content),
        first_ai_message,
    ]
}

print("ToolNode 输入（完整原始 state）：")
pprint(toolnode_input)
toolnode_result = tool_graph.invoke(toolnode_input)
print("\nToolNode 输出（完整原始 state）：")
pprint(toolnode_result)
print("\n最后两条原始消息：")
pprint(toolnode_result["messages"][-2:])

ToolNode 输入（完整原始 state）：
{'messages': [HumanMessage(content='请务必调用 add 工具计算 2 + 3，并调用 multiply 工具计算 4 × 5。两个工具都必须调用，不要自己心算；获得工具结果后再用中文汇总。', additional_kwargs={}, response_metadata={}, id='human-toolnode'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 385, 'total_tokens': 489, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 1}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'cd11239c-7e68-4b68-844c-cb2a8ac48988', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0343b-6f1f-76c2-a569-51b6267d276e-0', tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'call_00_ZRrAvbO8krCFgYKZUsUJ7560', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 4, 'b': 5}, 'id': 'cal

In [28]:
toolnode_messages = toolnode_result["messages"]
toolnode_tool_messages = [message for message in toolnode_messages if isinstance(message, ToolMessage)]

print("ToolNode 生成的完整 ToolMessage 列表：")
pprint(toolnode_tool_messages)

assert len(toolnode_tool_messages) == 2
assert {message.tool_call_id for message in toolnode_tool_messages} == set(requested_ids)
assert {message.name: json.loads(message.content) for message in toolnode_tool_messages} == {"add": 5, "multiply": 20}
assert isinstance(toolnode_messages[-1], ToolMessage)  # 没有模型节点，所以不会自动出现下一条 AIMessage。

continued_ai_message = tool_calling_model.invoke(toolnode_messages)
print("\n由应用显式触发的下一模型步骤（完整对象）：")
pprint(continued_ai_message)
assert isinstance(continued_ai_message, AIMessage)
assert not continued_ai_message.tool_calls
assert "5" in continued_ai_message.content and "20" in continued_ai_message.content
print("ToolNode 等价对照断言通过。")

ToolNode 生成的完整 ToolMessage 列表：
[ToolMessage(content='5', name='add', id='c2d64179-b88f-4778-9f1c-d7864ffee19e', tool_call_id='call_00_ZRrAvbO8krCFgYKZUsUJ7560'),
 ToolMessage(content='20', name='multiply', id='47738db8-76cb-4ebb-aba2-9d048d87ddcc', tool_call_id='call_01_6s6eEf0hYnPiuDuRXb5k9770')]

由应用显式触发的下一模型步骤（完整对象）：
AIMessage(content='根据工具计算结果：\n\n1. **add 工具**计算 2 + 3，结果为 **5**。\n2. **multiply 工具**计算 4 × 5，结果为 **20**。\n\n汇总：2 + 3 = 5，4 × 5 = 20。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 511, 'total_tokens': 572, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 127}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '4fc1297e-6e0c-445d-814b-c4cfb297e5f4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a034

## 3. ToolRuntime：读取运行环境，并通过 Command 更新状态

当工具参数标注为 `runtime: ToolRuntime[...]` 时，`ToolNode` 会在执行工具前注入它。`runtime` 不属于模型可填写的业务参数，所以不会出现在模型看到的工具 schema 中。

本例会实际读取：

- `runtime.state`：当前图状态快照，这里读取会员等级；
- `runtime.context`：本次调用的只读上下文，这里读取地区；
- `runtime.store`：编译图时注入的共享 Store，这里读取本地单价；
- `runtime.config`：本次运行配置，这里读取 `configurable.run_label`；
- `runtime.tool_call_id`：当前调用 ID，用来构造配对的 `ToolMessage`。

此外，`ToolRuntime` 还提供 `stream_writer`，可发出自定义流事件；本教程不启动流式消费，因此只说明边界而不使用它。State 可以通过返回值/`Command` 更新；context/config 不会因为修改局部对象而自动变成 state；store 需要调用其读写方法才能持久化数据。

In [29]:
class RuntimeState(TypedDict):
    messages: Annotated[list, add_messages]
    customer_tier: str
    reserved_quantity: int
    audit_log: Annotated[list[str], operator.add]


@dataclass(frozen=True)
class ShopContext:
    region: str


@tool
def lookup_local_stock(item: str) -> dict:
    """查询本地确定性库存。"""
    return {"item": item, "available": 8}


@tool
def command_without_tool_message(quantity: int, runtime: ToolRuntime[ShopContext, RuntimeState]) -> Command:
    """仅更新非 messages 字段，用于验证 ToolNode 的拒绝行为。"""
    return Command(update={"reserved_quantity": quantity})


@tool
def reserve_inventory(
    item: str,
    quantity: int,
    runtime: ToolRuntime[ShopContext, RuntimeState],
) -> Command:
    """读取运行时信息，预留库存，并显式返回匹配的 ToolMessage。"""
    tier = runtime.state["customer_tier"]
    region = runtime.context.region
    unit_price = runtime.store.get(("inventory",), item).value["unit_price"]
    run_label = runtime.config.get("configurable", {}).get("run_label")

    content = {
        "item": item,
        "quantity": quantity,
        "customer_tier": tier,
        "region": region,
        "unit_price": unit_price,
        "run_label": run_label,
        "observed_tool_call_id": runtime.tool_call_id,
    }
    return Command(
        update={
            "reserved_quantity": quantity,
            "audit_log": [f"reserve:{item}:{quantity}"],
            # Command 必须显式携带与当前调用配对的 ToolMessage。
            "messages": [
                ToolMessage(
                    content=json.dumps(content, ensure_ascii=False, sort_keys=True),
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [30]:
# 模型可见 schema 只包含业务参数，不包含 runtime。
visible_schema = reserve_inventory.tool_call_schema.model_json_schema()
print("模型可见的完整工具 schema：")
pprint(visible_schema)
print("\n模型可见字段：")
pprint(visible_schema["properties"])

assert set(visible_schema["properties"]) == {"item", "quantity"}
assert "runtime" not in visible_schema["properties"]
print("ToolRuntime 注入参数对模型不可见。")

模型可见的完整工具 schema：
{'description': '读取运行时信息，预留库存，并显式返回匹配的 ToolMessage。',
 'properties': {'item': {'title': 'Item', 'type': 'string'},
                'quantity': {'title': 'Quantity', 'type': 'integer'}},
 'required': ['item', 'quantity'],
 'title': 'reserve_inventory',
 'type': 'object'}

模型可见字段：
{'item': {'title': 'Item', 'type': 'string'},
 'quantity': {'title': 'Quantity', 'type': 'integer'}}
ToolRuntime 注入参数对模型不可见。


### 3.1 普通返回值：ToolNode 自动包装 ToolMessage

普通返回值不是 `Command`。`ToolNode` 会把它序列化为 `ToolMessage.content`，并自动带上工具名和当前调用 ID。下面先展示完整结果，再单独断言。

In [31]:
def compile_tool_graph(tool_object, *, store=None, context_schema=None):
    builder = StateGraph(RuntimeState, context_schema=context_schema)
    builder.add_node("tools", ToolNode([tool_object]))
    builder.add_edge(START, "tools")
    builder.add_edge("tools", END)
    return builder.compile(store=store)


def runtime_input(tool_name: str, args: dict, call_id: str) -> RuntimeState:
    return {
        "messages": [
            HumanMessage(id=f"human-{call_id}", content="执行本地库存操作。"),
            AIMessage(
                id=f"ai-{call_id}",
                content="",
                tool_calls=[{"name": tool_name, "args": args, "id": call_id, "type": "tool_call"}],
            ),
        ],
        "customer_tier": "gold",
        "reserved_quantity": 0,
        "audit_log": [],
    }


ordinary_graph = compile_tool_graph(lookup_local_stock)
ordinary_result = ordinary_graph.invoke(
    runtime_input("lookup_local_stock", {"item": "notebook"}, "call-ordinary-001")
)
print("普通返回值路径的完整最终 state：")
pprint(ordinary_result)
print("\n自动生成的完整 ToolMessage：")
pprint(ordinary_result["messages"][-1])

普通返回值路径的完整最终 state：
{'audit_log': [],
 'customer_tier': 'gold',
 'messages': [HumanMessage(content='执行本地库存操作。', additional_kwargs={}, response_metadata={}, id='human-call-ordinary-001'),
              AIMessage(content='', additional_kwargs={}, response_metadata={}, id='ai-call-ordinary-001', tool_calls=[{'name': 'lookup_local_stock', 'args': {'item': 'notebook'}, 'id': 'call-ordinary-001', 'type': 'tool_call'}], invalid_tool_calls=[]),
              ToolMessage(content='{"item": "notebook", "available": 8}', name='lookup_local_stock', id='be402c09-7a55-4e89-ab50-9369f8e86360', tool_call_id='call-ordinary-001')],
 'reserved_quantity': 0}

自动生成的完整 ToolMessage：
ToolMessage(content='{"item": "notebook", "available": 8}', name='lookup_local_stock', id='be402c09-7a55-4e89-ab50-9369f8e86360', tool_call_id='call-ordinary-001')


In [32]:
ordinary_message = ordinary_result["messages"][-1]
ordinary_payload = json.loads(ordinary_message.content)
print("ToolMessage 解码后的原始内容：")
pprint(ordinary_payload)

assert isinstance(ordinary_message, ToolMessage)
assert ordinary_message.name == "lookup_local_stock"
assert ordinary_message.tool_call_id == "call-ordinary-001"
assert ordinary_payload == {"item": "notebook", "available": 8}
assert ordinary_result["reserved_quantity"] == 0
print("普通返回值路径断言通过：ToolNode 自动创建了 ToolMessage。")

ToolMessage 解码后的原始内容：
{'available': 8, 'item': 'notebook'}
普通返回值路径断言通过：ToolNode 自动创建了 ToolMessage。


### 3.2 Command 只更新非 messages 字段：ToolNode 不会自动补 ToolMessage

当前版本会校验：当 `Command` 发往当前图时，每个工具调用都必须在 `Command.update["messages"]` 中提供一个 `tool_call_id` 匹配的 `ToolMessage`。因此“只更新 `reserved_quantity`”不会静默成功，也不会自动补消息，而是抛出 `ValueError`。

In [33]:
invalid_command_graph = compile_tool_graph(
    command_without_tool_message,
    context_schema=ShopContext,
)
invalid_command_input = runtime_input(
    "command_without_tool_message",
    {"quantity": 3},
    "call-invalid-command-001",
)

invalid_command_error = None
try:
    invalid_command_graph.invoke(
        invalid_command_input,
        context=ShopContext(region="CN-East"),
    )
except ValueError as exc:
    invalid_command_error = exc

print("捕获到的完整异常对象：")
pprint(invalid_command_error)
print("\n完整异常文本：")
print(str(invalid_command_error))

捕获到的完整异常对象：
ValueError('Expected to have a matching ToolMessage in Command.update for tool \'command_without_tool_message\', got: []. Every tool call (LLM requesting to call a tool) in the message history MUST have a corresponding ToolMessage. You can fix it by modifying the tool to return `Command(update={"messages": [ToolMessage("Success", tool_call_id=tool_call_id), ...]}, ...)`.')

完整异常文本：
Expected to have a matching ToolMessage in Command.update for tool 'command_without_tool_message', got: []. Every tool call (LLM requesting to call a tool) in the message history MUST have a corresponding ToolMessage. You can fix it by modifying the tool to return `Command(update={"messages": [ToolMessage("Success", tool_call_id=tool_call_id), ...]}, ...)`.


In [34]:
assert isinstance(invalid_command_error, ValueError)
assert "Expected to have a matching ToolMessage" in str(invalid_command_error)
assert "command_without_tool_message" in str(invalid_command_error)
assert invalid_command_input["reserved_quantity"] == 0  # 输入对象本身也未被原地修改。
assert len(invalid_command_input["messages"]) == 2
print("缺少 ToolMessage 的 Command 被拒绝，断言通过。")

缺少 ToolMessage 的 Command 被拒绝，断言通过。


### 3.3 Command 显式更新 messages：状态与 ToolMessage 一起写回

正确做法是在工具中用 `runtime.tool_call_id` 创建 `ToolMessage`，并把它和其他字段一起放进 `Command(update=...)`。LangGraph 随后按每个 state 字段声明的 reducer 应用更新。

In [35]:
store = InMemoryStore()
store.put(("inventory",), "notebook", {"unit_price": 12})

command_graph = compile_tool_graph(
    reserve_inventory,
    store=store,
    context_schema=ShopContext,
)
command_input = runtime_input(
    "reserve_inventory",
    {"item": "notebook", "quantity": 2},
    "call-reserve-001",
)
command_result = command_graph.invoke(
    command_input,
    config={"configurable": {"run_label": "local-demo"}},
    context=ShopContext(region="CN-East"),
)

print("Command 路径的完整输入 state：")
pprint(command_input)
print("\nCommand 应用后的完整最终 state：")
pprint(command_result)
print("\n显式写回的完整 ToolMessage：")
pprint(command_result["messages"][-1])

Command 路径的完整输入 state：
{'audit_log': [],
 'customer_tier': 'gold',
 'messages': [HumanMessage(content='执行本地库存操作。', additional_kwargs={}, response_metadata={}, id='human-call-reserve-001'),
              AIMessage(content='', additional_kwargs={}, response_metadata={}, id='ai-call-reserve-001', tool_calls=[{'name': 'reserve_inventory', 'args': {'item': 'notebook', 'quantity': 2}, 'id': 'call-reserve-001', 'type': 'tool_call'}], invalid_tool_calls=[])],
 'reserved_quantity': 0}

Command 应用后的完整最终 state：
{'audit_log': ['reserve:notebook:2'],
 'customer_tier': 'gold',
 'messages': [HumanMessage(content='执行本地库存操作。', additional_kwargs={}, response_metadata={}, id='human-call-reserve-001'),
              AIMessage(content='', additional_kwargs={}, response_metadata={}, id='ai-call-reserve-001', tool_calls=[{'name': 'reserve_inventory', 'args': {'item': 'notebook', 'quantity': 2}, 'id': 'call-reserve-001', 'type': 'tool_call'}], invalid_tool_calls=[]),
              ToolMessage(content='{"custo

In [36]:
command_message = command_result["messages"][-1]
command_payload = json.loads(command_message.content)
print("ToolMessage 解码后的全部运行时观测值：")
pprint(command_payload)

assert command_result["reserved_quantity"] == 2
assert command_result["audit_log"] == ["reserve:notebook:2"]
assert isinstance(command_message, ToolMessage)
assert command_message.name == "reserve_inventory"  # ToolNode 校验时会补齐工具名。
assert command_message.tool_call_id == "call-reserve-001"
assert command_payload == {
    "customer_tier": "gold",
    "item": "notebook",
    "observed_tool_call_id": "call-reserve-001",
    "quantity": 2,
    "region": "CN-East",
    "run_label": "local-demo",
    "unit_price": 12,
}
print("ToolRuntime 注入和 Command 显式消息更新断言全部通过。")

ToolMessage 解码后的全部运行时观测值：
{'customer_tier': 'gold',
 'item': 'notebook',
 'observed_tool_call_id': 'call-reserve-001',
 'quantity': 2,
 'region': 'CN-East',
 'run_label': 'local-demo',
 'unit_price': 12}
ToolRuntime 注入和 Command 显式消息更新断言全部通过。


## 4. 工具不经 ToolNode 执行：调用方承担协议责任

下面故意让普通图节点直接调用一个 LangChain 工具。工具返回的 `Command` 会被图运行时应用，但因为没有经过 `ToolNode`：

- 没有 ToolNode 帮忙把普通结果包装成 `ToolMessage`；
- 没有 ToolNode 做“每个调用必须配对 ToolMessage”的校验；
- 图状态中的待处理 `AIMessage.tool_calls` 仍然没有对应 `ToolMessage`。

这不是推荐的对话协议，而是边界验证：绕过 ToolNode 时，调用方必须自行注入所需运行时参数、执行工具、处理 `Command`，并保证消息历史完整。若工具需要 `ToolRuntime`，直接调用方还必须自己提供等价运行环境；通常应优先交给 ToolNode。

In [37]:
@tool
def direct_command_tool(ticket: str) -> Command:
    """返回不含 messages 的 Command，用于演示绕过 ToolNode 的边界。"""
    return Command(
        update={
            "reserved_quantity": 7,
            "audit_log": [f"direct:{ticket}"],
        }
    )


def direct_caller_node(state: RuntimeState) -> Command:
    # 这里是调用方直接执行工具，而不是把调用交给 ToolNode。
    return direct_command_tool.invoke({"ticket": "T-001"})


direct_builder = StateGraph(RuntimeState)
direct_builder.add_node("direct_caller", direct_caller_node)
direct_builder.add_edge(START, "direct_caller")
direct_builder.add_edge("direct_caller", END)
direct_graph = direct_builder.compile()

direct_input = runtime_input(
    "direct_command_tool",
    {"ticket": "T-001"},
    "call-direct-001",
)
direct_result = direct_graph.invoke(direct_input)

print("绕过 ToolNode 的完整最终 state：")
pprint(direct_result)
print("\n完整 messages（注意最后仍是带 tool_calls 的 AIMessage）：")
pprint(direct_result["messages"])

绕过 ToolNode 的完整最终 state：
{'audit_log': ['direct:T-001'],
 'customer_tier': 'gold',
 'messages': [HumanMessage(content='执行本地库存操作。', additional_kwargs={}, response_metadata={}, id='human-call-direct-001'),
              AIMessage(content='', additional_kwargs={}, response_metadata={}, id='ai-call-direct-001', tool_calls=[{'name': 'direct_command_tool', 'args': {'ticket': 'T-001'}, 'id': 'call-direct-001', 'type': 'tool_call'}], invalid_tool_calls=[])],
 'reserved_quantity': 7}

完整 messages（注意最后仍是带 tool_calls 的 AIMessage）：
[HumanMessage(content='执行本地库存操作。', additional_kwargs={}, response_metadata={}, id='human-call-direct-001'),
 AIMessage(content='', additional_kwargs={}, response_metadata={}, id='ai-call-direct-001', tool_calls=[{'name': 'direct_command_tool', 'args': {'ticket': 'T-001'}, 'id': 'call-direct-001', 'type': 'tool_call'}], invalid_tool_calls=[])]


In [38]:
direct_tool_messages = [
    message for message in direct_result["messages"] if isinstance(message, ToolMessage)
]
print("绕过 ToolNode 后的 ToolMessage 列表：")
pprint(direct_tool_messages)

assert direct_result["reserved_quantity"] == 7
assert direct_result["audit_log"] == ["direct:T-001"]
assert direct_tool_messages == []
assert direct_result["messages"][-1].tool_calls[0]["id"] == "call-direct-001"
print("边界断言通过：Command 更新被应用，但没有任何组件自动创建 ToolMessage。")

绕过 ToolNode 后的 ToolMessage 列表：
[]
边界断言通过：Command 更新被应用，但没有任何组件自动创建 ToolMessage。


## 5. add_messages reducer 与 tool_call_id

`messages: Annotated[list, add_messages]` 表示该字段不是简单覆盖：

- 新消息 ID 不同：追加；
- 新消息 ID 与旧消息相同：替换该消息；
- `Command(update={"messages": [...]})` 也遵循这个 reducer，不会绕过它。

`message.id` 与 `ToolMessage.tool_call_id` 是两个不同概念：

- `message.id` 标识消息自身，供 `add_messages` 判断追加还是替换；
- `tool_call_id` 引用 `AIMessage.tool_calls[*].id`，用于把工具结果配回具体调用。

一次 `AIMessage` 可以并列请求多个工具，所以不能只靠工具名或消息顺序配对，必须使用调用 ID。

In [39]:
original_messages = [HumanMessage(id="same-message-id", content="旧内容")]
replacement = HumanMessage(id="same-message-id", content="新内容")
appended = HumanMessage(id="new-message-id", content="追加内容")

after_replacement = add_messages(original_messages, [replacement])
after_append = add_messages(after_replacement, [appended])

print("原始消息列表：")
pprint(original_messages)
print("\n同 message.id 更新后的完整列表：")
pprint(after_replacement)
print("\n不同 message.id 追加后的完整列表：")
pprint(after_append)

assert len(after_replacement) == 1
assert after_replacement[0].content == "新内容"
assert len(after_append) == 2

paired_calls = first_ai_message.tool_calls
paired_results = created_tool_messages
pairing_table = [
    {
        "tool_name": call["name"],
        "AIMessage.tool_calls.id": call["id"],
        "ToolMessage.tool_call_id": result.tool_call_id,
        "ToolMessage.message_id": result.id,
    }
    for call, result in zip(paired_calls, paired_results, strict=True)
]
print("\n调用与结果的完整配对表：")
pprint(pairing_table)
assert all(
    row["AIMessage.tool_calls.id"] == row["ToolMessage.tool_call_id"]
    for row in pairing_table
)
print("add_messages 与调用 ID 配对断言通过。")

原始消息列表：
[HumanMessage(content='旧内容', additional_kwargs={}, response_metadata={}, id='same-message-id')]

同 message.id 更新后的完整列表：
[HumanMessage(content='新内容', additional_kwargs={}, response_metadata={}, id='same-message-id')]

不同 message.id 追加后的完整列表：
[HumanMessage(content='新内容', additional_kwargs={}, response_metadata={}, id='same-message-id'),
 HumanMessage(content='追加内容', additional_kwargs={}, response_metadata={}, id='new-message-id')]

调用与结果的完整配对表：
[{'AIMessage.tool_calls.id': 'call_00_ZRrAvbO8krCFgYKZUsUJ7560',
  'ToolMessage.message_id': 'c434df54-d7bc-4e1c-9012-ee6a6c518768',
  'ToolMessage.tool_call_id': 'call_00_ZRrAvbO8krCFgYKZUsUJ7560',
  'tool_name': 'add'},
 {'AIMessage.tool_calls.id': 'call_01_6s6eEf0hYnPiuDuRXb5k9770',
  'ToolMessage.message_id': '732782e0-17a3-4dd0-8c13-69ea66640839',
  'ToolMessage.tool_call_id': 'call_01_6s6eEf0hYnPiuDuRXb5k9770',
  'tool_name': 'multiply'}]
add_messages 与调用 ID 配对断言通过。


## 6. 当前版本源码核验

前面的运行结果已经覆盖四条路径。下面再读取当前环境中 `ToolNode._validate_tool_command` 的实际实现，完整输出源码并断言其中存在配对校验。该方法是私有实现细节，只用于锁定本教程所对应的版本；业务代码不应直接调用它。

In [40]:
validation_source = inspect.getsource(ToolNode._validate_tool_command)
print("当前环境 ToolNode._validate_tool_command 的完整源码：")
print(validation_source)

assert "Expected to have a matching ToolMessage" in validation_source
assert 'message.tool_call_id == call["id"]' in validation_source
assert "if updated_command.graph is None and not has_matching_tool_message" in validation_source
print("源码断言通过：当前图上的 Command 必须显式携带匹配 ToolMessage。")

当前环境 ToolNode._validate_tool_command 的完整源码：
    def _validate_tool_command(
        self,
        command: Command,
        call: ToolCall,
        input_type: Literal["list", "dict", "tool_calls"],
    ) -> Command:
        if isinstance(command.update, dict):
            # input type is dict when ToolNode is invoked with a dict input
            # (e.g. {"messages": [AIMessage(..., tool_calls=[...])]})
            if input_type not in ("dict", "tool_calls"):
                msg = (
                    "Tools can provide a dict in Command.update only when using dict "
                    f"with '{self._messages_key}' key as ToolNode input, "
                    f"got: {command.update} for tool '{call['name']}'"
                )
                raise ValueError(msg)

            updated_command = deepcopy(command)
            state_update = cast("dict[str, Any]", updated_command.update) or {}
            messages_update = state_update.get(self._messages_key, [])
        elif isinstanc

## 结论矩阵

| 执行方式 | 工具返回值 | `ToolMessage` 是否自动产生 | 结果 |
|---|---|---:|---|
| 经 `ToolNode` | 普通值 / 字符串 / 字典 | 是 | ToolNode 包装并写入 `messages` |
| 经 `ToolNode` | `Command`，只更新非 `messages` 字段 | 否 | 当前图上校验失败，抛出 `ValueError` |
| 经 `ToolNode` | `Command`，显式更新 `messages` 且 ID 匹配 | 工具显式提供，ToolNode 校验 | state 与消息按 reducer 一起更新 |
| 不经 `ToolNode` | `Command` | 否 | 图可应用 Command，但调用方必须维护工具调用协议 |

最终要点：

- `ToolNode` 是工具执行节点，不是模型节点；
- `ToolRuntime` 由 ToolNode 注入，模型不可见，适合访问当前 state 与运行依赖；
- 普通返回值和 `Command` 走不同路径，不能假设 ToolNode 会为所有返回值自动补 `ToolMessage`；
- 使用 `Command` 时，显式构造 `ToolMessage(tool_call_id=runtime.tool_call_id)` 是保持消息历史合法的关键；
- `add_messages` 管理消息合并，`tool_call_id` 管理调用—结果配对，两者职责不同。